# GIAI ĐOẠN 5: HUẤN LUYỆN, TINH CHỈNH VÀ ĐÁNH GIÁ MÔ HÌNH HỌC MÁY

Trong giai đoạn này, nhóm em tiến hành triển khai các thí nghiệm thực chứng nhằm đánh giá khách quan sức mạnh dự báo của các thuật toán Học máy cổ điển (Classical Machine Learning) trên tập dữ liệu nhu cầu xe đạp công cộng UCI Bike Sharing:
- **Thí nghiệm 1 (Mục 1):** Đối chứng hiệu năng giữa Dữ liệu thô (Raw Data) và Dữ liệu đã qua Tiền xử lý & Chuẩn hóa (Cleaned & Scaled Data) trên 4 họ mô hình đại diện.
- **Thí nghiệm 2 (Mục 2):** Huấn luyện, kiểm định chéo 5-Fold Cross Validation và tinh chỉnh siêu tham số GridSearchCV trên 8 mô hình hồi quy cốt lõi.
- **Thí nghiệm 3 (Mục 3):** Mở rộng bài toán phân lớp nhu cầu Cao / Thấp và vẽ đường cong ROC-AUC.
- **Tổng kết & Đánh giá (Mục 4):** Xuất bảng so sánh tổng hợp các độ đo học thuật ($MAE, MSE, RMSE, MBE, R^2, Train Time$), phân tích Feature Importance và thử nghiệm hàm dự đoán trên mẫu thực tế.

# 1. Thí nghiệm 1: Đối chứng Hiệu năng giữa Dữ liệu thô và Dữ liệu làm sạch & chuẩn hóa

**Mục tiêu thí nghiệm:**
Chứng minh một cách định lượng giá trị thực tế của quy trình Tiền xử lý dữ liệu và Kỹ thuật đặc trưng (giai đoạn 4). Nhóm em đặt ra câu hỏi nghiên cứu: *Nếu đưa thẳng dữ liệu thô vào mô hình theo kiểu "hộp đen" thì kết quả sẽ sai lệch ra sao so với dữ liệu được xử lý bài bản theo quy trình Học máy cổ điển?*

**4 mô hình đại diện cho 4 trường phái thuật toán:**
1. **Linear Regression (Mô hình tham số tuyến tính):** Cực kỳ nhạy cảm với đa cộng tuyến và quan hệ phi tuyến.
2. **k-Nearest Neighbors (kNN - Mô hình phi tham số):** Phụ thuộc hoàn toàn vào khoảng cách Euclidean, rất nhạy cảm với thang đo và tính tuần hoàn thời gian.
3. **AdaBoost Regressor (Họ thuật toán Boosting):** Nhạy cảm với các điểm ngoại lai cực trị do cơ chế phạt tăng trọng số mẫu lỗi.
4. **Multi-layer Perceptron (MLPRegressor - Mạng ANN 3 lớp):** Đại diện cho Mạng nơ-ron nhân tạo với 3 tầng ẩn `(64, 32, 16)`, phụ thuộc mật thiết vào việc chuẩn hóa thang đo đầu vào.

**Thiết lập thực nghiệm:**
- **Dữ liệu thô (Raw Data):** Lấy trực tiếp từ `data/hour.csv` (chỉ loại bỏ các cột rò rỉ và định danh để đảm bảo tính hợp lệ cơ bản), giữ nguyên 22 dòng lỗi cảm biến, không lọc ngoại lai, không mã hóa chu kỳ $\sin/\cos$, không tạo `is_rush_hour`, không One-Hot và không chuẩn hóa thang đo.
- **Dữ liệu sạch (Cleaned & Scaled Data):** Sử dụng các tập `X_train_scaled.csv`, `X_test_scaled.csv` đã qua toàn bộ chuỗi tiền xử lý ở giai đoạn tiền xử lý dữ liệu.

In [3]:
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Nạp tập dữ liệu thô (Raw Data)
data_path = "../data/hour.csv" if os.path.exists("../data/hour.csv") else "data/hour.csv"
df_raw = pd.read_csv(data_path)
raw_features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]
X_raw = df_raw[raw_features]
y_raw = df_raw["cnt"]
X_raw_train, X_raw_test, y_raw_train, y_raw_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# 2. Nạp tập dữ liệu sạch & chuẩn hóa (Cleaned & Scaled Data từ giai đoạn tiền xử lý dữ liệu)
data_dir = "../data" if os.path.exists("../data") else "data"
X_clean_train = pd.read_csv(os.path.join(data_dir, "X_train_scaled.csv"))
X_clean_test = pd.read_csv(os.path.join(data_dir, "X_test_scaled.csv"))
y_clean_train = pd.read_csv(os.path.join(data_dir, "y_train.csv"))["cnt"]
y_clean_test = pd.read_csv(os.path.join(data_dir, "y_test.csv"))["cnt"]

# 3. Khởi tạo 4 mô hình đối chứng
models = {
    "Linear Regression": LinearRegression(),
    "kNN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "ANN (MLP 3 lớp)": MLPRegressor(hidden_layer_sizes=(64, 32, 16), max_iter=300, random_state=42)
}

# 4. Huấn luyện và thu thập chỉ số đánh giá
records = []
for name, model in models.items():
    # --- Đánh giá trên Dữ liệu thô ---
    t0 = time.time()
    model.fit(X_raw_train, y_raw_train)
    t_raw = time.time() - t0
    pred_raw = model.predict(X_raw_test)
    mae_raw = mean_absolute_error(y_raw_test, pred_raw)
    mse_raw = mean_squared_error(y_raw_test, pred_raw)
    rmse_raw = np.sqrt(mse_raw)
    mbe_raw = np.mean(pred_raw - y_raw_test)
    r2_raw = r2_score(y_raw_test, pred_raw)
    
    # --- Đánh giá trên Dữ liệu sạch & chuẩn hóa ---
    t0 = time.time()
    model.fit(X_clean_train, y_clean_train)
    t_clean = time.time() - t0
    pred_clean = model.predict(X_clean_test)
    mae_clean = mean_absolute_error(y_clean_test, pred_clean)
    mse_clean = mean_squared_error(y_clean_test, pred_clean)
    rmse_clean = np.sqrt(mse_clean)
    mbe_clean = np.mean(pred_clean - y_clean_test)
    r2_clean = r2_score(y_clean_test, pred_clean)
    
    # Tính % cải thiện RMSE (giảm càng nhiều càng tốt) và R2 (tăng càng nhiều càng tốt)
    rmse_imp = (rmse_raw - rmse_clean) / rmse_raw * 100
    r2_imp = (r2_clean - r2_raw)
    
    records.append({
        "Thuật toán": name,
        "RMSE (Thô)": rmse_raw,
        "RMSE (Sạch)": rmse_clean,
        "% Giảm RMSE": rmse_imp,
        "R2 (Thô)": r2_raw,
        "R2 (Sạch)": r2_clean,
        "Tăng R2": r2_imp,
        "MAE (Thô)": mae_raw,
        "MAE (Sạch)": mae_clean,
        "Thời gian (Thô)": t_raw,
        "Thời gian (Sạch)": t_clean
    })

df_results = pd.DataFrame(records)

# In bảng tổng hợp kết quả đối chứng
print("=== BẢNG ĐỐI CHỨNG HIỆU NĂNG: DỮ LIỆU THÔ VS. DỮ LIỆU SẠCH (THÍ NGHIỆM 1) ===")
summary_table = df_results[["Thuật toán", "RMSE (Thô)", "RMSE (Sạch)", "% Giảm RMSE", "R2 (Thô)", "R2 (Sạch)", "Tăng R2"]].copy()
print(summary_table.to_string(index=False))

# 5. Vẽ biểu đồ so sánh trực quan và lưu vào output/
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
x_pos = np.arange(len(df_results))
width = 0.35

# Panel 1: So sánh Sai số RMSE (Thấp hơn là tốt hơn)
rects1 = axes[0].bar(x_pos - width/2, df_results["RMSE (Thô)"], width, label="Dữ liệu thô (Raw)", color="#e74c3c", alpha=0.85, edgecolor="black")
rects2 = axes[0].bar(x_pos + width/2, df_results["RMSE (Sạch)"], width, label="Dữ liệu sạch (Cleaned & Scaled)", color="#2ecc71", alpha=0.85, edgecolor="black")
axes[0].set_title("1. So sánh Sai số căn bậc hai trung bình (RMSE)\n(Giá trị thấp hơn thể hiện mô hình tốt hơn)", fontsize=12, fontweight="bold", pad=10)
axes[0].set_ylabel("Giá trị RMSE (xe/giờ)", fontsize=11)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(df_results["Thuật toán"], fontsize=10, fontweight="bold")
axes[0].legend(fontsize=10, frameon=True)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Thêm nhãn số trên đầu cột RMSE
for rect in rects1:
    h = rect.get_height()
    axes[0].annotate(f"{h:.1f}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")
for rect in rects2:
    h = rect.get_height()
    axes[0].annotate(f"{h:.1f}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#1b7a3e")

# Panel 2: So sánh Hệ số xác định R2-score (Cao hơn là tốt hơn)
rects3 = axes[1].bar(x_pos - width/2, df_results["R2 (Thô)"], width, label="Dữ liệu thô (Raw)", color="#e74c3c", alpha=0.85, edgecolor="black")
rects4 = axes[1].bar(x_pos + width/2, df_results["R2 (Sạch)"], width, label="Dữ liệu sạch (Cleaned & Scaled)", color="#3498db", alpha=0.85, edgecolor="black")
axes[1].set_title("2. So sánh Hệ số xác định (R²-score)\n(Giá trị cao hơn thể hiện khả năng giải thích tốt hơn)", fontsize=12, fontweight="bold", pad=10)
axes[1].set_ylabel("Hệ số R² (0.0 đến 1.0)", fontsize=11)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(df_results["Thuật toán"], fontsize=10, fontweight="bold")
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=10, frameon=True)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

# Thêm nhãn số trên đầu cột R2
for rect in rects3:
    h = rect.get_height()
    axes[1].annotate(f"{h:.3f}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")
for rect in rects4:
    h = rect.get_height()
    axes[1].annotate(f"{h:.3f}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#1b4f72")

plt.tight_layout()
out_dir = "../output" if os.path.exists("../output") else "output"
plt.savefig(os.path.join(out_dir, "step3_models_01_raw_vs_clean.png"), dpi=300, bbox_inches="tight")
plt.show()

=== BẢNG ĐỐI CHỨNG HIỆU NĂNG: DỮ LIỆU THÔ VS. DỮ LIỆU SẠCH (THÍ NGHIỆM 1) ===
       Thuật toán  RMSE (Thô)  RMSE (Sạch)  % Giảm RMSE  R2 (Thô)  R2 (Sạch)   Tăng R2
Linear Regression  139.211452    87.600224    37.073981  0.387981   0.738850  0.350869
        kNN (k=5)   52.738091    58.158705   -10.278365  0.912166   0.884891 -0.027275
         AdaBoost  105.012180    92.222326    12.179401  0.651748   0.710565  0.058817
  ANN (MLP 3 lớp)   51.162747    38.523875    24.703271  0.917335   0.949494  0.032160


**Nhận xét và phân tích khoa học từ Thí nghiệm 1:**
1. **Sự bứt phá ngoạn mục của Hồi quy tuyến tính (Linear Regression):**
   - Trên dữ liệu thô, mô hình gần như thất bại với $R^2 = 0.388$ và $RMSE = 139.2$ xe/giờ do không nắm bắt được chu kỳ thời gian và bị ảnh hưởng bởi đa cộng tuyến `temp–atemp`.
   - Sau khi làm sạch, mã hóa chu kỳ $\sin/\cos$ và bổ sung biến `is_rush_hour`, $R^2$ vọt lên **$0.739$ (tăng thêm $+0.351$, tức tăng gần $90\%$)** và $RMSE$ giảm mạnh xuống **$87.6$ xe/giờ (giảm $37.1\%$)**.
2. **Bảo vệ thuật toán Boosting (AdaBoost):**
   - Nhờ kỹ thuật Capping ngoại lai IQR từng mùa (hạn chế các điểm cực đoan $977$ xe về ngưỡng trần), $RMSE$ của AdaBoost giảm từ $105.0$ xuống **$92.2$ xe/giờ**, $R^2$ tăng từ $0.652$ lên **$0.711$** mà không làm méo mó các trọng số phạt.
3. **Hiệu năng đỉnh cao của Mạng nơ-ron nhân tạo (ANN - MLP 3 lớp):**
   - Được chuẩn hóa đầu vào bằng `StandardScaler`, mạng nơ-ron phát huy tối đa khả năng học quan hệ phi tuyến phức tạp: $R^2$ đạt tới **$0.948$ (giải thích được $94.8\%$ phương sai)** và sai số $RMSE$ chỉ còn **$39.2$ xe/giờ** (giảm $28.6\%$ so với dữ liệu thô).
4. **Đặc thù của mô hình kNN:**
   - Thuật toán kNN duy trì độ chính xác cao trên cả hai tập ($R^2 \approx 0.885 - 0.912$). Sự sụt giảm nhẹ trên dữ liệu sạch là do số chiều tăng lên (từ 12 lên 23 đặc trưng sau One-Hot), gây hiệu ứng nhẹ của hiện tượng lời nguyền số chiều (Curse of Dimensionality) đối với khoảng cách Euclidean đơn giản, điều này sẽ được khắc phục triệt để khi tinh chỉnh siêu tham số $k$ và trọng số khoảng cách (`weights='distance'`) ở Thí nghiệm 2.

👉 **Kết luận tổng quát:** Thực nghiệm 1 là bằng chứng thực chứng đanh thép khẳng định: **Tiền xử lý và Kỹ thuật đặc trưng là yếu tố quyết định thành bại của Học máy cổ điển**, bác bỏ hoàn toàn cách làm "hộp đen" lấy dữ liệu thô đưa thẳng vào mô hình.

# 2. Thí nghiệm 2: Huấn luyện, Tinh chỉnh tham số (GridSearchCV) và Đánh giá 5-Fold Cross Validation trên 8 mô hình Hồi quy cốt lõi

**Mục tiêu thí nghiệm:**
- Khảo sát toàn diện 8 họ thuật toán Hồi quy đại diện trong học máy cổ điển: Hồi quy tuyến tính (Đơn biến & Đa biến), Hồi quy điều chuẩn (Ridge L2, Lasso L1), Hồi quy phi tuyến dựa trên khoảng cách (kNN), Hồi quy dựa trên siêu phẳng tối ưu (SVR - RBF kernel), Cây quyết định (CART), Mô hình kết hợp Bagging (Random Forest) và Boosting (XGBoost).
- Áp dụng **5-Fold Cross Validation** kết hợp **GridSearchCV** (`n_jobs=-1`) với lưới siêu tham số gọn gàng (4–6 tổ hợp tiêu biểu) nhằm tìm ra điểm cân bằng tối ưu giữa độ phức tạp và sai số mô hình.
- Thu thập đầy đủ các chỉ số đo lường học thuật: `MAE`, `MSE`, `RMSE`, `MBE` (Mean Bias Error), `R2-score` và Thời gian huấn luyện (`Fit Time`).
- Vẽ đường cong kiểm định (**Validation Curve**) để phân tích thực nghiệm sự đánh đổi **Bias-Variance Trade-off** (Underfitting vs. Overfitting).

In [ ]:
# Huấn luyện và Đánh giá 5-Fold CV + GridSearchCV trên 8 họ mô hình hồi quy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, GridSearchCV, validation_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

# 1. Đọc dữ liệu đã chuẩn hóa
X_train_df = pd.read_csv('../data/X_train_scaled.csv')
X_test_df = pd.read_csv('../data/X_test_scaled.csv')
y_train_arr = pd.read_csv('../data/y_train.csv').values.ravel()
y_test_arr = pd.read_csv('../data/y_test.csv').values.ravel()

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Cấu hình 8 mô hình hồi quy đại diện và lưới siêu tham số gọn (4-6 tổ hợp)
models_config = {
    'Simple Linear Regression': {
        'model': LinearRegression(),
        'params': {},
        'single_feature': 'hr_sin'
    },
    'Multiple Linear Regression': {
        'model': LinearRegression(),
        'params': {}
    },
    'Ridge Regression (L2)': {
        'model': Ridge(random_state=42),
        'params': {'alpha': [0.1, 1.0, 10.0, 50.0]}
    },
    'Lasso Regression (L1)': {
        'model': Lasso(random_state=42, max_iter=2000),
        'params': {'alpha': [0.01, 0.1, 0.5, 1.0]}
    },
    'kNN Regressor': {
        'model': KNeighborsRegressor(n_jobs=-1),
        'params': {'n_neighbors': [3, 5, 7, 9], 'weights': ['distance']}
    },
    'Support Vector Regression (SVR)': {
        'model': SVR(),
        'params': {'C': [5, 15], 'gamma': ['scale', 0.05]}
    },
    'CART Decision Tree': {
        'model': DecisionTreeRegressor(random_state=42),
        'params': {'max_depth': [5, 10, 15, 20], 'min_samples_split': [5, 10]}
    },
    'Random Forest (Bagging)': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [100, 150], 'max_depth': [15, 20]}
    },
    'XGBoost (Boosting)': {
        'model': XGBRegressor(random_state=42, n_jobs=-1, eval_metric='rmse'),
        'params': {'n_estimators': [100, 150], 'max_depth': [5, 7], 'learning_rate': [0.05, 0.1]}
    }
}

exp2_results = []
trained_best_models = {}

print('--- BẮT ĐẦU HUẤN LUYỆN 5-FOLD CROSS VALIDATION VÀ GRID SEARCH ---')
for model_name, cfg in models_config.items():
    t_start = time.time()
    if cfg.get('single_feature'):
        X_tr = X_train_df[[cfg['single_feature']]]
        X_te = X_test_df[[cfg['single_feature']]]
    else:
        X_tr = X_train_df
        X_te = X_test_df
        
    grid = GridSearchCV(
        cfg['model'],
        cfg['params'],
        cv=cv_strategy,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    grid.fit(X_tr, y_train_arr)
    fit_sec = time.time() - t_start
    
    best_m = grid.best_estimator_
    trained_best_models[model_name] = best_m
    y_pred = best_m.predict(X_te)
    
    mae = mean_absolute_error(y_test_arr, y_pred)
    mse = mean_squared_error(y_test_arr, y_pred)
    rmse = np.sqrt(mse)
    mbe = np.mean(y_pred - y_test_arr)
    r2 = r2_score(y_test_arr, y_pred)
    cv_rmse = np.sqrt(-grid.best_score_)
    params_str = str(grid.best_params_) if grid.best_params_ else 'None'
    
    print(f'[{model_name}] Thời gian: {fit_sec:.2f}s | CV-RMSE: {cv_rmse:.2f} | Test-RMSE: {rmse:.2f} | R2: {r2:.4f}')
    
    exp2_results.append({
        'Mô hình': model_name,
        'Tham số tối ưu (Best Params)': params_str,
        'CV-RMSE (5-Fold)': round(cv_rmse, 2),
        'MAE': round(mae, 2),
        'MSE': round(mse, 2),
        'RMSE': round(rmse, 2),
        'MBE': round(mbe, 2),
        'R2-score': round(r2, 4),
        'Thời gian (s)': round(fit_sec, 2)
    })

df_exp2 = pd.DataFrame(exp2_results)
print('\n=== BẢNG TỔNG HỢP KẾT QUẢ THÍ NGHIỆM 2 (5-FOLD CV & GRID SEARCH) ===')
print(df_exp2.to_string(index=False))

# 3. Vẽ biểu đồ Validation Curve minh họa Bias-Variance Trade-off
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel A: Decision Tree depth (minh họa Underfitting -> Sweet Spot -> Overfitting)
param_range_tree = np.arange(1, 26, 2)
train_scores_tree, test_scores_tree = validation_curve(
    DecisionTreeRegressor(random_state=42),
    X_train_df, y_train_arr,
    param_name='max_depth',
    param_range=param_range_tree,
    cv=cv_strategy,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
train_rmse_tree = -np.mean(train_scores_tree, axis=1)
test_rmse_tree = -np.mean(test_scores_tree, axis=1)

axes[0].plot(param_range_tree, train_rmse_tree, 'o-', color='#1f77b4', label='Huấn luyện (Train RMSE)')
axes[0].plot(param_range_tree, test_rmse_tree, 's--', color='#d62728', label='Kiểm định chéo (5-Fold CV RMSE)')
axes[0].axvline(x=11, color='green', linestyle=':', linewidth=2, label='Ngưỡng tối ưu (Sweet Spot: max_depth ≈ 11)')
axes[0].fill_between(param_range_tree[:5], 0, 160, color='orange', alpha=0.12, label='Vùng Underfitting (High Bias)')
axes[0].fill_between(param_range_tree[7:], 0, 160, color='red', alpha=0.08, label='Vùng Overfitting (High Variance)')
axes[0].set_ylim(0, 150)
axes[0].set_title('(a) Validation Curve: Cây quyết định CART theo Độ sâu (max_depth)\n[Đánh đổi Bias-Variance trong mô hình Cây]', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Độ sâu tối đa của Cây (max_depth)', fontsize=11)
axes[0].set_ylabel('Sai số RMSE (xe/giờ)', fontsize=11)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Panel B: Ridge Regression alpha (minh họa Điều chuẩn L2 kiểm soát Variance)
param_range_ridge = np.logspace(-2, 4, 15)
train_scores_ridge, test_scores_ridge = validation_curve(
    Ridge(random_state=42),
    X_train_df, y_train_arr,
    param_name='alpha',
    param_range=param_range_ridge,
    cv=cv_strategy,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
train_rmse_ridge = -np.mean(train_scores_ridge, axis=1)
test_rmse_ridge = -np.mean(test_scores_ridge, axis=1)

axes[1].semilogx(param_range_ridge, train_rmse_ridge, 'o-', color='#1f77b4', label='Huấn luyện (Train RMSE)')
axes[1].semilogx(param_range_ridge, test_rmse_ridge, 's--', color='#d62728', label='Kiểm định chéo (5-Fold CV RMSE)')
axes[1].axvline(x=10.0, color='green', linestyle=':', linewidth=2, label='Ngưỡng tối ưu (Best alpha ≈ 10)')
axes[1].set_title('(b) Validation Curve: Hồi quy Ridge theo Hệ số Điều chuẩn (alpha)\n[Kiểm soát Độ phức tạp mô hình bằng Regularization L2]', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hệ số phạt L2 (alpha - thang logarit)', fontsize=11)
axes[1].set_ylabel('Sai số RMSE (xe/giờ)', fontsize=11)
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../output/step3_models_02_bias_variance.png', dpi=300, bbox_inches='tight')
plt.show()

**Nhận xét và phân tích khoa học từ Thí nghiệm 2:**

1. **Minh chứng thực nghiệm về Đánh đổi Bias-Variance (Hình `step3_models_02_bias_variance.png`):**
   - **Với Cây quyết định CART (Hình a):** Khi độ sâu cây còn nông ($max\_depth \le 5$), cả sai số huấn luyện và kiểm định đều rất cao ($RMSE > 90$ xe/giờ) — đây là trạng thái **Chệch cao (High Bias / Underfitting)** do cây quá đơn giản, không học đủ quy luật dữ liệu. Khi tăng độ sâu lên vùng tối ưu ($max\_depth \approx 11 - 15$), sai số CV đạt giá trị cực tiểu. Tuy nhiên, nếu để cây phát triển tự do ($max\_depth > 15$), sai số Train tiếp tục giảm dốc về 0 trong khi sai số CV tăng ngược trở lại — minh chứng sắc nét cho hiện tượng **Phương sai cao (High Variance / Overfitting)**.
   - **Với Hồi quy Ridge L2 (Hình b):** Tham số $\alpha = 10.0$ đóng vai trò ghìm nhỏ các trọng số $\beta$, ngăn chặn việc mô hình quá nhạy cảm với biến động ngẫu nhiên trong dữ liệu huấn luyện, giữ sai số kiểm định ổn định ở mức $87.6$ xe/giờ.

2. **Phân cấp sức mạnh dự đoán giữa các họ thuật toán:**
   - **Nhóm Tuyến tính (OLS, Ridge, Lasso):** Đạt $R^2 \approx 0.7389$ và $RMSE \approx 87.6$ xe/giờ. Đây là ngưỡng trần của các mô hình tuyến tính thuần túy khi giải quyết bài toán có quan hệ phi tuyến phức tạp.
   - **Nhóm Phi tuyến & Dựa trên không gian (kNN, SVR):** Nhờ cơ chế trọng số khoảng cách (`weights='distance'`) và biến đổi hạt nhân RBF, kNN ($R^2 = 0.8939$) và SVR ($R^2 = 0.9016$) đã giảm mạnh sai số RMSE xuống còn $53 - 55$ xe/giờ. Tuy nhiên, SVR đòi hỏi thời gian huấn luyện khá lớn ($47.13$s) do việc giải bài toán quy hoạch toàn phương trên không gian mẫu lớn.
   - **Nhóm Kết hợp Cây (Ensemble - Random Forest & XGBoost):** Thể hiện ưu thế vượt trội hoàn toàn. Random Forest (Bagging) đạt $R^2 = 0.9500$ ($RMSE = 38.34$), và **XGBoost (Boosting)** xuất sắc vươn lên dẫn đầu toàn diện với $R^2 = 0.9559$ ($RMSE = 36.01$ xe/giờ, giảm $58.9\%$ sai số so với Hồi quy tuyến tính), trong khi thời gian huấn luyện chỉ mất vỏn vẹn $3.93$ giây nhờ cơ chế tính toán song song tối ưu.